# MEI Consistency Checks

This notebook was migrated from `camat_corpus`. It checks the page-level MEI files in `Demo/finished_material`, prepares normalized copies, combines them, and loads source-trackable findings into DataFrames for proofreading. The workflow now imports the packaged `camat` helpers, so a separate `scripts/` directory is not required.

Automatically useful checks for preparing a digital edition include:

- XML/MEI hygiene: well-formed XML, MEI namespace/version, duplicate `xml:id`, broken internal references such as `@startid`, `@endid`, `@facs`, `@plist`.
- Facsimile alignment: measures with missing or broken `@facs`, measure zones with invalid coordinates, unused measure zones.
- Measure/staff structure: sequential measure numbers, missing or extra staff numbers against active `staffDef`, missing layers, duplicate layer numbers.
- Rhythmic consistency: every staff/layer should fill the active meter according to written `dur`/`dots`, including `note`, `rest`, `chord`, `space`, `mRest`, and `mSpace`; underfull layers without `space` are highlighted because they often mark omitted voices or historical notation problems.
- Musical primitives: missing written durations, suspicious chord duration encoding, invalid pitch names/octaves.
- Instrumentation/staff consistency: empty/generated labels such as `System 1` or `Prt1`, inconsistent labels/abbreviations/MIDI instruments for the same staff number across files, and staff-count/staff-number-set variants across selected files.
- Musical terms: empty directions, unanchored tempo/direction terms, and spelling or punctuation variants such as `Adagio`/`Adagio.` across the corpus.

The report columns are designed to be source-trackable: `file`, `line`, `measure_n`, `staff_n`, `layer_n`, and `xml_id` point back to the MEI location.

In [ ]:
from pathlib import Path
import sys
import urllib.error
import urllib.request
import xml.etree.ElementTree as ET

# Find the CAMAT checkout whether Jupyter starts in the repository root or notebooks/.
_here = Path.cwd().resolve()
CAMAT_ROOT = next(
    (path for path in (_here, *_here.parents) if (path / "camat" / "__init__.py").is_file()),
    None,
)
if CAMAT_ROOT is None:
    raise RuntimeError("Run this notebook from a CAMAT source checkout.")
if str(CAMAT_ROOT) not in sys.path:
    sys.path.insert(0, str(CAMAT_ROOT))

# The copied corpus inputs remain in the sibling checkout by default. Change ROOT for another corpus.
_default_corpus_root = CAMAT_ROOT.parent / "camat_corpus"
ROOT = _default_corpus_root if _default_corpus_root.is_dir() else CAMAT_ROOT

from camat.mei_consistency_workflow import (
    annotate_mei_from_report,
    annotation_filter_description,
    combine_meis,
    load_report,
    make_unique_xml_id_copies,
    report_summary,
    resolve_mei_inputs,
    run_checker,
    source_snippet,
    strip_ppq_copies,
)

MEI_NS = "http://www.music-encoding.org/ns/mei"
IIIF_CHECK_TIMEOUT = 15
CHECK_IIIF_LINKS = True  # Network check; set False for an offline run.


def facsimile_graphic_targets(mei_path):
    root = ET.parse(mei_path).getroot()
    graphics = root.findall(f".//{{{MEI_NS}}}facsimile//{{{MEI_NS}}}graphic")
    if not graphics:
        graphics = []
        for facsimile in root.iter():
            if facsimile.tag.rsplit("}", 1)[-1] != "facsimile":
                continue
            graphics.extend(
                element
                for element in facsimile.iter()
                if element.tag.rsplit("}", 1)[-1] == "graphic"
            )
    return [graphic.get("target") for graphic in graphics]


def iiif_url_available(url, timeout=IIIF_CHECK_TIMEOUT):
    request = urllib.request.Request(
        url,
        headers={
            "Range": "bytes=0-0",
            "User-Agent": "mei-consistency-check/1.0",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            status = response.status
        return 200 <= status < 400, f"HTTP {status}"
    except urllib.error.HTTPError as exc:
        return False, f"HTTP {exc.code}"
    except urllib.error.URLError as exc:
        return False, str(exc.reason)
    except TimeoutError:
        return False, "timeout"


def check_facsimile_iiif_links(mei_files):
    issues = []
    for mei_path in mei_files:
        try:
            targets = facsimile_graphic_targets(mei_path)
        except ET.ParseError as exc:
            issues.append(f"{mei_path}: XML parse error: {exc}")
            continue

        if not targets:
            issues.append(f"{mei_path}: missing <facsimile>/<graphic @target>")
            continue

        for target in targets:
            if not target:
                issues.append(f"{mei_path}: missing <graphic @target>")
                continue
            if not target.startswith(("http://", "https://")) or "/iiif/" not in target.lower():
                issues.append(f"{mei_path}: non-IIIF facsimile target: {target}")
                continue
            available, message = iiif_url_available(target)
            if not available:
                issues.append(f"{mei_path}: unavailable IIIF facsimile target ({message}): {target}")
    return issues


# Add directories and/or individual MEI files here. Directories are expanded recursively.
MEI_INPUTS = [
    "Demo/finished_material/bsb00023199_00126_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00127_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00128_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00129_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00130_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00131_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00132_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00133_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00134_facs_zones.mei",
    "Demo/finished_material/bsb00023199_00135_facs_zones.mei",
]

UNIQUE_XML_IDS = True
UNIQUE_MEI_DIR = ROOT / "Demo" / "finished_material_unique_ids"

STRIP_PPQ = True
STRIPPED_MEI_DIR = ROOT / "Demo" / "finished_material_no_ppq"
REPORT_DIR = STRIPPED_MEI_DIR if STRIP_PPQ else UNIQUE_MEI_DIR if UNIQUE_XML_IDS else ROOT / "Demo" / "finished_material"
CSV_OUT = REPORT_DIR / "mei_consistency_report.csv"
JSON_OUT = REPORT_DIR / "mei_consistency_report.json"

CHECK_PPQ = False  # set True to compare optional import/playback @dur.ppq values against written durations

RENUMBER_COMBINED_MEASURES = True
NUMBER_COMBINED_MEASURE_ZONES = True
NORMALIZE_SINGLE_LAYER_NUMBERS = True

# Optional naming override for the final *_full.mei file.
# If True, the first combined scoreDef is treated as the canonical staff/staffGrp naming source.
KEEP_ORIGINAL_COMBINED_STAFF_NAMES = True
# Use None to leave existing staff labels untouched, or provide {"1": "...", "2": "..."}.
# Example: COMBINED_STAFF_NAMES = {"1": "Violino.", "2": "Viola da gamba.", "3": "Cembalo.", "4": "...", "5": "..."}
# COMBINED_STAFF_GROUP_LABEL writes a label onto the nested staffGrp when one exists.
COMBINED_STAFF_NAMES = None
COMBINED_STAFF_ABBREVIATIONS = None
COMBINED_STAFF_GROUP_LABEL = None
COMBINED_STAFF_GROUP_ABBREVIATION = None
COMBINED_STAFF_NAMES_ONLY_AT_START = True

# Annotation controls for the final *_full_annotated.mei file.
# Keep max_annotations <= 100 for mei-friend's default annotation display limit.
ANNOTATE_SEVERITIES = {"error"}          # examples: {"error"}, {"error", "warning"}
ANNOTATE_CATEGORIES = None               # None means all; example: {"references", "rhythm"}
ANNOTATE_CHECKS = None                   # None means all; example: {"broken_internal_reference"}
ANNOTATE_EXCLUDE_CHECKS = set()          # example: {"measure_sequence"}
MAX_ANNOTATIONS = 100

INITIAL_MEI_FILES = resolve_mei_inputs(MEI_INPUTS, ROOT)
if not INITIAL_MEI_FILES:
    raise FileNotFoundError("No .mei files found in MEI_INPUTS")
facsimile_iiif_issues = (
    check_facsimile_iiif_links(INITIAL_MEI_FILES) if CHECK_IIIF_LINKS else []
)
if facsimile_iiif_issues:
    issue_text = "\n".join(f"- {issue}" for issue in facsimile_iiif_issues)
    raise RuntimeError(f"Missing or unavailable IIIF facsimile links:\n{issue_text}")

PREPARED_MEI_FILES = INITIAL_MEI_FILES
ACTIVE_MEI_FILES = PREPARED_MEI_FILES
print(f"Selected {len(INITIAL_MEI_FILES)} MEI file(s).")
if CHECK_IIIF_LINKS:
    print(f"Verified IIIF facsimile links for {len(INITIAL_MEI_FILES)} MEI file(s).")
else:
    print("IIIF availability check skipped.")
print("Annotation filter:", annotation_filter_description(
    severities=ANNOTATE_SEVERITIES,
    categories=ANNOTATE_CATEGORIES,
    checks=ANNOTATE_CHECKS,
    exclude_checks=ANNOTATE_EXCLUDE_CHECKS,
    max_annotations=MAX_ANNOTATIONS,
))


In [ ]:
if UNIQUE_XML_IDS:
    unique_result = make_unique_xml_id_copies(INITIAL_MEI_FILES, UNIQUE_MEI_DIR)
    PREPARED_MEI_FILES = unique_result.files
    print(f"Wrote {len(PREPARED_MEI_FILES)} ID-unique MEI file(s) into {UNIQUE_MEI_DIR}.")
    print(unique_result.message)
else:
    PREPARED_MEI_FILES = INITIAL_MEI_FILES
    print("XML ID uniqueness preparation bypassed.")

ACTIVE_MEI_FILES = PREPARED_MEI_FILES


In [ ]:
if STRIP_PPQ:
    ppq_result = strip_ppq_copies(PREPARED_MEI_FILES, STRIPPED_MEI_DIR)
    ACTIVE_MEI_FILES = ppq_result.files
    print(f"Stripped files written into {STRIPPED_MEI_DIR}.")
    print(ppq_result.message)
else:
    ACTIVE_MEI_FILES = PREPARED_MEI_FILES
    print("PPQ stripping bypassed; checks will run against the prepared MEI files.")

REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Active input contains {len(ACTIVE_MEI_FILES)} MEI file(s).")


In [ ]:
run_checker(
    ACTIVE_MEI_FILES,
    root=ROOT,
    csv_out=CSV_OUT,
    json_out=JSON_OUT,
    check_ppq=CHECK_PPQ,
)


In [ ]:
df = load_report(CSV_OUT)
summary = report_summary(df)
summary


In [ ]:
base_stem = ACTIVE_MEI_FILES[0].stem.removesuffix("_noppq").removesuffix("_unique_ids")
COMBINED_MEI_OUT = REPORT_DIR / f"{base_stem}_full.mei"
combine_result = combine_meis(
    ACTIVE_MEI_FILES,
    COMBINED_MEI_OUT,
    renumber_measures=RENUMBER_COMBINED_MEASURES,
    number_measure_zones=NUMBER_COMBINED_MEASURE_ZONES,
    normalize_single_layer_numbers=NORMALIZE_SINGLE_LAYER_NUMBERS,
    staff_names=COMBINED_STAFF_NAMES,
    staff_abbreviations=COMBINED_STAFF_ABBREVIATIONS,
    staff_group_label=COMBINED_STAFF_GROUP_LABEL,
    staff_group_abbreviation=COMBINED_STAFF_GROUP_ABBREVIATION,
    keep_original_staff_names=KEEP_ORIGINAL_COMBINED_STAFF_NAMES,
    staff_names_only_at_start=COMBINED_STAFF_NAMES_ONLY_AT_START,
)

print(f"Combined {len(ACTIVE_MEI_FILES)} MEI file(s) into {combine_result.path}.")
print(f"Duplicate xml:id value(s) in combined file: {len(combine_result.duplicate_ids)}")
print(f"Skipped <expansion> element(s) while flattening page sections: {combine_result.skipped_expansions}")
print(f"Renumbered measure @n values: {combine_result.renumbered_measures}")
print(f"Numbered referenced measure zones: {combine_result.numbered_zones}")
print(f"Normalized single-layer @n values: {combine_result.normalized_single_layers}")
print(f"Staff name/abbr overrides written: {combine_result.staff_names_written}")
print(f"Staff group name/abbr overrides written: {combine_result.staff_group_names_written}")
print(f"Later scoreDef staff labels removed: {combine_result.later_staff_names_removed}")
if combine_result.duplicate_ids:
    print(combine_result.duplicate_ids[:20])


In [ ]:
COMBINED_CSV_OUT = COMBINED_MEI_OUT.with_name(f"{COMBINED_MEI_OUT.stem}_consistency_report.csv")
COMBINED_JSON_OUT = COMBINED_MEI_OUT.with_name(f"{COMBINED_MEI_OUT.stem}_consistency_report.json")

run_checker(
    [COMBINED_MEI_OUT],
    root=ROOT,
    csv_out=COMBINED_CSV_OUT,
    json_out=COMBINED_JSON_OUT,
    check_ppq=CHECK_PPQ,
)

combined_df = load_report(COMBINED_CSV_OUT)
combined_summary = report_summary(combined_df).sort_values(["severity", "category", "check"])
combined_issue_rows = combined_df[[
    "severity", "category", "check", "line", "measure_n", "staff_n", "layer_n", "xml_id", "message", "expected", "actual", "context"
]].head(200)

try:
    display(combined_summary)
    display(combined_issue_rows)
except NameError:
    print(combined_summary.to_string(index=False))
    print(combined_issue_rows.to_string(index=False))


In [ ]:
ANNOTATED_FULL_MEI_OUT = COMBINED_MEI_OUT.with_name(f"{COMBINED_MEI_OUT.stem}_annotated.mei")
annotation_result = annotate_mei_from_report(
    COMBINED_MEI_OUT,
    combined_df,
    ANNOTATED_FULL_MEI_OUT,
    severities=ANNOTATE_SEVERITIES,
    categories=ANNOTATE_CATEGORIES,
    checks=ANNOTATE_CHECKS,
    exclude_checks=ANNOTATE_EXCLUDE_CHECKS,
    max_annotations=MAX_ANNOTATIONS,
)

print(f"Wrote annotated full MEI: {annotation_result.path}")
print(f"Created {annotation_result.created} consistency annotation(s); skipped {annotation_result.skipped} finding(s) without an anchorable xml_id.")
if annotation_result.capped:
    print(f"Annotation output was capped at MAX_ANNOTATIONS={MAX_ANNOTATIONS}.")
